# Roadmap & Rationale — Online Retail II Forecasting

**This notebook is internal documentation, not part of the final submission.** It records *why* every design call was made and *what* still has to be built. Status badges per section:

- ✅ **IMPLEMENTED** — code lives in `src/` or `agent/`
- 🔧 **PARTIAL** — partial implementation; gaps noted
- 📋 **PLANNED** — pseudocode here, no code yet

Pipeline overview:
```
raw xlsx → load → split sales/returns → weekly aggregate → return rate
         → calendar/lags/price/demand-class → embedding → clustering
         → rolling-origin selection → 12-week forecast → revenue translation
         → agent parquet → n8n agent
```

## 1. Loading — both Excel sheets — ✅

Implementation: `src/tools/data_loader.py::load_raw_data`. The prior team only loaded sheet 1 (the pandas default), missing all of 2009-2010. We use `sheet_name=None` and concatenate with a `SourceSheet` column. `dtype={"Invoice": str, "StockCode": str}` is critical — alphanumeric codes like `85123A` are real and must not be coerced to int.

## 2. Drop rules — ✅

Implementation: `src/tools/cleaning.py::split_sales_returns`. Rationale comments live in the module so the file is self-documenting (this notebook is not shipped).

Filters in order:
1. Drop nulls on `InvoiceDate` / `StockCode` / `Description`.
2. Cast `Invoice`, `StockCode` → string. `Description` → `.strip()`.
3. Keep only `StockCode` matching `^\d{5}[A-Za-z]?$` — drops admin/fee rows (POST, M, D, BANK CHARGES, AMAZONFEE, CRUK, …).
4. Drop sentinel descriptions (`?`, `damaged`, `lost`, `check`, `wrongly coded`).
5. Drop `Price <= 0` (5 negative + 6 202 zero rows: accounting adjustments / samples / gifts).
6. Build Mon-aligned `Week`.
7. Split into `sales` (target) and `returns` (feature only). C-invoices and `Quantity<0` go to `returns`; the rest stays in `sales`.

**The two big calls** (and why we rejected the alternatives):

| Choice | Rejected alternative | Why we picked ours |
|---|---|---|
| Drop `Quantity<0` from target, build `return_rate_Nw` feature | Net via weekly sum | Net produces `y<0` when return lags purchase across weeks — breaks NegBin (DeepAR), Tweedie (LightGBM), `log1p` (LSTM/iTransformer). Clipping to 0 wastes the netting. |
| Use returns as feature, ignore C-invoice matching | FIFO match `(CustomerID, StockCode)` | 22.8% of rows have null `Customer ID` → matching fails on a quarter of data. O(n²) cost to recover bias on <0.4% of rows post-filter. |

## 3. Feature engineering

### 3.1 Target — ✅
`y = Quantity` summed per `(StockCode, Week)`. Per-SKU series reindexed to a continuous Mon axis with `fillna(0)` (intermittency is signal). `build_series_for_sku` does the reindex; `aggregate_weekly_sku` does the panel-level sum.

### 3.2 Calendar — ✅
`add_calendar_features`: `week_of_year`, `month`, `quarter`, `year`, sin/cos of woy and month, `holiday_uk` (via `holidays` package, country GB), `is_christmas_window`. Christmas dominates this gift-ware dataset, so it has its own flag.

### 3.3 Lag / rolling — ✅
`add_lag_rolling_features`: lags `[1,2,4,8,13,26,52]`, rolling mean/std on windows `[4,13,26]`. Used by LightGBM-recursive natively; can be passed as `dynamic_real` to DeepAR / NS-Transformer.

### 3.4 Price — ✅
`add_price_features`: `price_w` (median per SKU/week), `price_rel_sku` (price_w / SKU lifetime median), `price_pct_change_w`, `promo_flag` (price_rel_sku < 0.85).

### 3.4-bis Returns — ✅
`return_rate_features`: rolling sum of returns / rolling sum of sales over (4, 13)-week windows. No CustomerID needed → 100% data coverage.

### 3.5 Demand classification (Syntetos-Boylan) — ✅
`demand_classification`: ADI (Average Demand Interval) and CV² over non-zero demand → `smooth / erratic / intermittent / lumpy`. Used to route SKUs to the right model (intermittent → Croston-SBA per-cluster, smooth → SARIMAX/Prophet).

### 3.6 Static commercial profile — ✅
`commercial_profile`: `price_median`, `mean_basket_size` (proxy for wholesale vs retail), `n_unique_customers`, `country_uk_share`, `price_tier` (quantile bin). Enters as `static_real` in DeepAR / NS-Transformer and as clustering features.

### 3.7 Description embedding — ✅
`embed_sku_descriptions` calls `gemini-embedding-2-preview` with `task_type="CLUSTERING"`, `output_dimensionality=768`, L2-normalized. Cached to parquet — first run ~$0.01, subsequent runs free.

## 4. Models — what each one needs

Status legend per model: ✅ implemented (callable from `src.modelling`), 🔧 partial, 📋 skeleton with TODO.

| Model | Status | Where | Loss |
|---|---|---|---|
| Naive (last-value baseline) | ✅ | `src/modelling/naive.py` | n/a |
| SARIMAX `(1,1,1)(1,1,0,52)` + exog | ✅ | `src/modelling/sarimax.py` | MLE |
| Prophet + UK Bank Holidays + regressors | ✅ | `src/modelling/prophet_model.py` | MAP |
| LightGBM recursive (lags 1..8) | ✅ | `src/modelling/lightgbm_recursive.py` | **Tweedie**, var_power=1.2 |
| DeepAR (gluonts) | 📋 | `src/modelling/deepar.py` (skeleton + pseudocode) | Negative-Binomial |
| Non-Stationary Transformer | 🔧 | `src/modelling/ns_transformer/{architecture,train}.py` | Huber on log1p |

Local models (Naive, SARIMAX, Prophet, LightGBM) follow the contract `(train: pd.Series, horizon: int) -> np.ndarray` and plug into `default_registry()`. Global models (DeepAR, NS-Transformer) train once on the panel and expose a per-SKU adapter via `deepar_factory(...)` / `predict_ns_transformer(...)`.

In [ ]:
# Pseudocode: extending the registry to include a global model
# from src.modelling import default_registry
# from src.modelling.deepar import DeepARWrapper, deepar_factory
#
# wrapper = DeepARWrapper(context_length=52, prediction_length=12)
# wrapper.fit(weekly_sku, static_features=commercial_profile_with_cluster, dynamic_features=...)
# registry = {**default_registry(), "DeepAR": deepar_factory(wrapper)}
# select_best_model(weekly_sku, skus, registry, n_folds=3, test_size=12, ...)

## 5. Clustering (HDBSCAN) — ✅

Implementation: `src/tools/clustering.py::cluster_skus`.

Pipeline:
```
embeddings (768)         ──UMAP(32, cosine)──┐
demand profile (3 cols)  ──StandardScaler─────┼──► concat ──► HDBSCAN(min_cluster_size=20, euclidean)
commercial profile (4 cols) ──StandardScaler──┘
```

**Why HDBSCAN over KMeans**: KMeans assumes spherical equal-variance clusters — false here (long tail on demand profile). HDBSCAN finds variable-density clusters and isolates outliers as `-1` (noise bucket = rare SKUs we don't force into a group).

**Validation**: silhouette is informative but the true validator is downstream WMAPE — clusters that don't reduce WMAPE are noise.

In [ ]:
# Pseudocode: end-to-end clustering call
# from src.tools import (split_sales_returns, aggregate_weekly_sku, demand_classification,
#                        commercial_profile, embed_sku_descriptions, cluster_skus, cluster_summary)
#
# sales, returns = split_sales_returns(df)
# weekly_sku = aggregate_weekly_sku(sales)
# emb = embed_sku_descriptions(sales, cache_path="data/processed/sku_desc_emb.parquet")
# demand = demand_classification(weekly_sku)
# commercial = commercial_profile(sales)
# clusters = cluster_skus(emb, demand, commercial, umap_dim=32, min_cluster_size=20)
# print(cluster_summary(clusters))

## 6. WMAPE strategy

$$\text{WMAPE} = \frac{\sum_i |y_i - \hat{y}_i|}{\sum_i |y_i|}$$

Volume-weighted: high-volume SKUs dominate the global score.

### 6.1 Loss alignment — 🔧 partial
- DeepAR: `negative-binomial` (gluonts default for counts) — 📋
- LightGBM: `objective="tweedie"`, `tweedie_variance_power=1.2` — ✅ (implemented in `lightgbm_recursive.py`)
- NS-Transformer: `HuberLoss` on `log1p(qty)` — ✅

### 6.2 Problem segmentation — 📋
Pseudocode for the planned routing logic:
```python
for sku in skus:
    cls = demand_class[sku]
    vol_tier = volume_tier[sku]
    if cls in ("intermittent", "lumpy"):
        # Croston-SBA per cluster — Transformer overfits and predicts ~0
        forecast = croston_per_cluster(sku)
    elif vol_tier == "top":
        forecast = ensemble([deepar(sku), ns_transformer(sku)])
    else:
        forecast = best_local(sku)  # SARIMAX/Prophet/LightGBM
```

### 6.3 Validation — ✅ rolling-origin
`rolling_origin_evaluate` in `src/tools/evaluation.py`. **Decision**: 3 folds × 12-week test, expanding train origin, `min_train=70` (≥1.35 seasons for SARIMAX s=52). With ~106 weeks of data, fold 1 train = 70w, fold 2 train = 82w, fold 3 train = 94w. Fold 1 doubles as validation for model selection; folds 2–3 are held-out test.

### 6.4 Ensemble — 📋
Pseudocode:
```python
# Per SKU, weight each model by 1/MAE on the validation fold
weights = {m: 1 / mae[m] for m in registry}
Z = sum(weights.values())
y_ensemble = sum(w * y_hat[m] for m, w in weights.items()) / Z
```
Errors of local (SARIMAX, Prophet) and global (DeepAR, NS-Transformer) models are decorrelated; expect 1–3 WMAPE points improvement over the best single model.

### 6.5 Bias correction per cluster — 📋
$$\hat{y}_{final} = a_c \cdot \hat{y}_{model} + b_c$$
Fit `(a_c, b_c)` on the validation fold per cluster. Free 0.5–1.5 points typically.

### 6.6 Sanity baselines — 🔧 partial
- Naive (last-value) — ✅
- Seasonal-naive (`y_t = y_{t-52}`) — 📋 not yet, easy to add as `src/modelling/seasonal_naive.py`
- Rolling mean 4w — 📋 not yet

Pseudocode for the missing baselines:
```python
def seasonal_naive(train, horizon, season=52):
    if len(train) < season:
        return naive(train, horizon)
    return train.values[-season:][:horizon]  # repeat last full season

def rolling_mean_4w(train, horizon):
    base = float(train.tail(4).mean())
    return np.repeat(base, horizon)
```

## 7. Agent (n8n) — 🔧 partial

Implementation: `agent/forecast_service.py` (prompt → forecast text), `agent/server.py` (FastAPI exposing `/forecast`). The n8n workflow JSON is in `agent/n8n_workflow_example.json`.

Sequence:
1. Playground writes `data/processed/agent_summary.parquet` and `agent_horizon.parquet`.
2. `uvicorn agent.server:app --port 8000` boots; the server reads the parquet on startup.
3. n8n AI Agent calls the `forecast_lookup` HTTP tool with the user message.
4. Server returns formatted forecast text; the agent relays to the user.

Still TODO: API auth header (the example workflow assumes localhost; add a bearer token before exposing publicly), proper logging, monitoring.

## 8. Suggested workflow checklist

1. ✅ Cleaning + sales/returns split
2. ✅ Weekly aggregation + return rate features
3. ✅ Calendar / lags / price / demand-class / commercial features
4. ✅ Gemini description embeddings (parquet-cached)
5. ✅ HDBSCAN clustering → `cluster_id`
6. 🔧 Baselines: Naive ✅, Seasonal-Naive 📋, Rolling-Mean-4w 📋
7. ✅ SARIMAX seasonal + Prophet + LightGBM Tweedie via `default_registry()`
8. 📋 DeepAR (wire `DeepARWrapper.fit/forecast`)
9. 🔧 NS-Transformer (architecture + train pipeline ready; integrate with rolling-origin)
10. 📋 Per-cluster bias correction + ensemble